<h2>Readability and Unfairness in Terms of Service Documents</h2>
CSE 595 Course Project

Kristine McLaughlin

<h4>1. Data Preprocessing</h4>

In [2]:
import os
import re
import pandas as pd

In [34]:
# Extract annotated clauses from tagged documents to build dataset
folder_path = 'TaggedDocuments_142'
data = []

for file in os.listdir(folder_path):
    file_path = os.path.join(folder_path, file)
    with open(file_path, "r", encoding="utf-8") as f:
        document_text = f.read()

    # Handling nested tags with a stack
    tag_stack = []
    buffer_text = ""
    clause_start = 0 # index where the clause text starts

    # Get all tags in document (open and close)
    tag_pattern = re.compile(r"<(/?)([a-zA-Z]+)(\d+)>") # <potential backslash, then letters, then a number>
    for tag in tag_pattern.finditer(document_text):
        tag_type = tag.group(1)  # '' for open, '/' for close
        category = tag.group(2)
        num = int(tag.group(3))
        tag_start, tag_end = tag.span() # start is index of tag start <, end is index after >

        # Opening tag - add tag to stack
        if tag_type == "":
            tag_stack.append((category, num))

        # Closing tag
        if tag_type == "/" and (category, num) in tag_stack:
            inner_text = document_text[clause_start:tag_start].strip().lower()

            if inner_text:  # only record if there’s text
                for category, num in tag_stack:
                    data.append({
                        "file_name": file,
                        "category": category,
                        "fairness_score": num,
                        "text": inner_text
                    })

            # Remove the matching tag from the stack
            if tag_stack and tag_stack[-1] == (category, num):
                tag_stack.pop()
            else:
                # Handle mismatched nesting by removing first occurrence
                if (category, num) in tag_stack:
                    tag_stack.remove((category, num))
        
        # Update where the text (should) start
        clause_start = tag_end

# Turn array into df
data = pd.DataFrame(data, columns=["file_name", "category", "fairness_score", "text"])
print(f"Extracted {len(data)} annotated clauses from {len(os.listdir(folder_path))} files")
print(data.head())

Extracted 3554 annotated clauses from 142 files
        file_name category  fairness_score  \
0  MyHeritage.xml      use               2   
1  MyHeritage.xml      use               2   
2  MyHeritage.xml       ch               2   
3  MyHeritage.xml      use               2   
4  MyHeritage.xml       cr               2   

                                                text  
0  please read them carefully, because by using t...  
1  if you do not agree with any provision in this...  
2  we reserve the right to modify any provision h...  
3  you agree to be bound to any changes to this a...  
4  we will not edit or monitor user-provided cont...  


In [35]:
# Manual check: Save to csv to investigate
data.to_csv("data.csv", index=False)

In [36]:
# Remove entries with fairness score of 4
data = data[data['fairness_score'] != 4]

In [37]:
# Get statistics on data
data.describe()

,fairness_score
count,3546.000000
mean,2.143542
std,0.463000
min,1.000000
25%,2.000000
50%,2.000000
75%,2.000000
max,3.000000


In [38]:
data['fairness_score'].value_counts()

fairness_score
2    2713
3     671
1     162
Name: count, dtype: int64

In [45]:
# First testing dataset just be the clearly fair (1) and clearly unfair (3) clauses
clear_fair = data[data['fairness_score'] == 1]
clear_unfair = data[data['fairness_score'] == 3]
data_clear = pd.concat([clear_fair, clear_unfair])

# Add binary fair/unfair value. 0 = fair, 1 = unfair
data_clear['unfair'] = (data_clear['fairness_score'] == 3).astype(int)

In [46]:
data_clear.describe()

,fairness_score,unfair
count,833.000000,833.000000
mean,2.611044,0.805522
std,0.792072,0.396036
min,1.000000,0.000000
25%,3.000000,1.000000
50%,3.000000,1.000000
75%,3.000000,1.000000
max,3.000000,1.000000


In [47]:
data_clear['unfair'].value_counts()

unfair
1    671
0    162
Name: count, dtype: int64

<h4>2. Calculate Baselines</h4>

In [ ]:
import textstat

In [ ]:
def get_readability_score(clause):
    flesch_kincaid = textstat.flesch_kincaid_grade(clause) # 0-100. Higher = easier.
    flesch_ease = textstat.flesch_reading_ease(clause) # 0-100. Higher = easier
    gunning_fog = textstat.gunning_fog(clause) # 0-20. Years of education needed. Higher = more difficult
    smog = textstat.smog_index(clause) # 0- ~20? No upper bound. Years of education needed, based heavily on # polysyllabic words
